# 🚗 AI Vehicle Service, Repair & Insurance Intelligence System
## Industrial End-to-End Pure Machine Learning Benchmark

**Architecture Highlights:**
- **Domain**: Automotive Service Operations, Fleet Management & Auto Insurance
- **Paradigms**: Pure Machine Learning (Tabular Regression, Binary Classification, NLP Diagnostic Retrieval)
- **Algorithms**: Linear Regression, Logistic Regression, Random Forest, XGBoost
- **Datasets**: 4 Structured Repositories (Main Repair/Towing, Used Car Specs, Fraud Oracle, Diagnostic Assistant)
- **Evaluation Framework**: MAE, MSE, RMSE, R², Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb

# Set visualization aesthetics
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries and visualization environments loaded successfully.')

## 1. Data Ingestion & Schema Inspection
Loading cleaned tabular dataset of vehicle repair, maintenance, and insurance events.

In [2]:
DATA_PATH = os.path.join('..', 'datasets', 'cleaned', 'main_cleaned.csv')
df = pd.read_csv(DATA_PATH, nrows=50000, low_memory=False)
print(f'Cleaned Dataset Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA)
Visualizing distributions of Total Cost, Service Duration, and Technician Ratings.

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['Total Cost'], kde=True, ax=axes[0], color='#06b6d4')
axes[0].set_title('Repair Cost Distribution ($)', fontsize=14)

sns.histplot(df['Service Duration Hours'], kde=True, ax=axes[1], color='#10b981')
axes[1].set_title('Service Duration Hours Distribution', fontsize=14)

sns.countplot(data=df, x='Vehicle Type', ax=axes[2], palette='viridis')
axes[2].set_title('Workorders by Vehicle Type', fontsize=14)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Top Replaced Parts Multi-Hot Analysis

In [4]:
part_cols = [c for c in df.columns if c.startswith('part_') and c != 'Parts_Clean']
part_sums = df[part_cols].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=part_sums.values, y=[p.replace('part_', '').replace('_', ' ').title() for p in part_sums.index], palette='mako')
plt.title('Top 10 Most Frequently Serviced / Replaced Automotive Parts', fontsize=14)
plt.xlabel('Frequency Across Workorders')
plt.show()

## 4. Pure ML Model Benchmarking: Repair Cost (Regression)
Comparing **Linear Regression**, **Random Forest Regressor**, and **XGBoost Regressor**.

In [5]:
import joblib
fe_path = os.path.join('..', 'models', 'fe_pipeline.pkl')
sys.path.insert(0, os.path.abspath('..'))
from preprocessing.transformers import FrequencyEncoder

fe_dict = joblib.load(fe_path)
preprocessor = fe_dict['pipeline']
feature_cols = fe_dict['feature_cols']

X = preprocessor.transform(df[feature_cols])
y_cost = df['Total Cost'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y_cost, test_size=0.2, random_state=42)

lr = LinearRegression().fit(X_tr, y_tr)
rf = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
xg = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.08, random_state=42).fit(X_tr, y_tr)

models = {'Linear Regression': lr, 'Random Forest': rf, 'XGBoost': xg}
for name, m in models.items():
    pred = m.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2 = r2_score(y_te, pred)
    print(f'{name:20s} | MAE: ${mae:.2f} | RMSE: ${rmse:.2f} | R2 Score: {r2:.4f}')

## 5. Pure ML Model Benchmarking: Insurance Claim (Binary Classification)
Comparing **Logistic Regression**, **Random Forest Classifier**, and **XGBoost Classifier**.

In [6]:
y_ins = df['Insurance Claim Used_flag'].astype(int).values
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X, y_ins, test_size=0.2, random_state=42, stratify=y_ins)

clf_log = LogisticRegression(max_iter=500).fit(X_tr_i, y_tr_i)
clf_rf = RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1).fit(X_tr_i, y_tr_i)
clf_xgb = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.08, eval_metric='logloss', random_state=42).fit(X_tr_i, y_tr_i)

clfs = {'Logistic Regression': clf_log, 'Random Forest': clf_rf, 'XGBoost': clf_xgb}
for name, m in clfs.items():
    pred = m.predict(X_te_i)
    prob = m.predict_proba(X_te_i)[:, 1]
    acc = accuracy_score(y_te_i, pred)
    roc = roc_auc_score(y_te_i, prob)
    print(f'{name:22s} | Accuracy: {acc*100:.2f}% | ROC-AUC: {roc:.4f}')

## 6. Diagnostic NLP & Rule-Based Recommendation Agent
Testing symptom NLP classification and expert replacement rules.

In [7]:
diag_model = joblib.load(os.path.join('..', 'models', 'diag_model.pkl'))
with open(os.path.join('..', 'models', 'diag_rules.json'), 'r', encoding='utf-8') as f:
    rules = json.load(f)

test_symptom = 'Headlights flicker and battery light on dashboard'
predicted_cat = diag_model.predict([test_symptom])[0]
print(f'Input Symptom: "{test_symptom}"')
print(f'AI Classification: {predicted_cat}')
print(f'Recommended Fixes: {rules["categories"].get(predicted_cat, {}).get("recommended_fixes", [])[:3]}')
print(f'Replace Parts:     {rules["categories"].get(predicted_cat, {}).get("replace_parts", [])[:3]}')